# BoliSense: Dialect-Aware Sentiment Analysis for Haryanvi Text

**Course**: Natural Language Processing (CSE Final Year)  
**Tagline**: *Understanding Regional Language Beyond Keywords*  
**Approach**: Classical NLP and Linguistic Feature Engineering  
**Proposed Model**: BoliSense Hybrid Classifier  

---

## 1. Project Overview & Objectives
This notebook demonstrates the complete end-to-end NLP workflow for **BoliSense**, an empirical research project investigating sentiment analysis on informal, non-standardized Roman-script Haryanvi text.

### Key Steps:
1. **Corpus Loading & Validation**: Clean and de-duplicate the raw dataset into 546 unique sentences.
2. **Zero-Leakage Partitioning**: 70% Train (382), 15% Validation (82), 15% Test (82).
3. **Feature Engineering**: Dialect phonetic normalization, Word TF-IDF, Character n-grams (3-5), windowed negation scope, and contrastive clause parsing.
4. **Model Benchmarking**: Train and evaluate 6 models (Majority, Naive Bayes, Word LogReg, Char LogReg, Linear SVM, and BoliSense Hybrid).
5. **Ablation Study & Robustness**: Assess linguistic components and evaluate on the 65-sample minimal-pair challenge dataset.
6. **Explainability**: Local linear feature contribution calculations ($x_i \times w_i$).

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on Python path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.dataset import load_raw_dataset, clean_and_split_dataset, load_splits, load_challenge_dataset
from src.features import DialectNormalizer, extract_linguistic_features, get_hybrid_feature_union
from src.models import train_all_models, evaluate_model, MODEL_BUILDERS
from src.explainability import BoliSenseInferenceEngine

print("Libraries and BoliSense modules loaded successfully.")

## 2. Dataset Preparation and Partitioning
We clean the dataset to remove exact duplicates and perform stratified 70-15-15 splitting.

In [ ]:
raw_df = load_raw_dataset()
train_df, val_df, test_df = clean_and_split_dataset(raw_df)

print(f"Raw dataset records: {len(raw_df)}")
print(f"Cleaned unique sentences: {len(train_df) + len(val_df) + len(test_df)}")
print(f"Train split: {len(train_df)} | Val split: {len(val_df)} | Test split: {len(test_df)}")
print("\nClass balance in test split:")
print(test_df["label"].value_counts().rename({0: "Negative", 1: "Positive"}))

## 3. Linguistic Feature Extraction & Normalization Demo
Demonstrating dialect phonetic normalization, negation detection (*konya*, *koni*), and contrastive clause parsing (*par*).

In [ ]:
normalizer = DialectNormalizer()
sample_text = "yo kaam ghna badhiya konya par agla step theek se"
normalized = normalizer.normalize(sample_text)
features = extract_linguistic_features(sample_text)

print(f"Original:   {sample_text}")
print(f"Normalized: {normalized}")
print("\nExtracted Linguistic Features:")
for k, v in features.items():
    if v != 0:
        print(f"  - {k}: {v}")

## 4. Train Models
We train all 6 models on the training partition ($N=382$).

In [ ]:
trained_models = train_all_models(train_df)
print("All 6 models trained successfully.")

## 5. Comparative Evaluation on Held-Out Test Set (N=82)

In [ ]:
results = []
for name, model in trained_models.items():
    res = evaluate_model(model, test_df["sentence"], test_df["label"])
    results.append({
        "Model": name,
        "Accuracy": res["Accuracy"],
        "Macro_Precision": res["Macro_Precision"],
        "Macro_Recall": res["Macro_Recall"],
        "Macro_F1": res["Macro_F1"],
        "Positive_F1": res["Positive_F1"]
    })

comparison_df = pd.DataFrame(results)
display(comparison_df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
plot_data = comparison_df.melt(id_vars="Model", value_vars=["Accuracy", "Macro_F1"], var_name="Metric", value_name="Score")
sns.barplot(data=plot_data, x="Model", y="Score", hue="Metric", palette=["#2b5c8f", "#d95f02"], ax=ax)
ax.set_ylim(0, 1.05)
ax.set_title("BoliSense Model Comparison (Test Set N=82)", fontsize=12, fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 6. Linguistic Challenge Evaluation & Explainability Demo
Stress-testing models on the 65-sample minimal-pair challenge set and viewing transparent linear feature contributions.

In [ ]:
chal_df = load_challenge_dataset()
chal_results = []
for name, model in trained_models.items():
    preds = model.predict(chal_df["sentence"])
    acc = (preds == chal_df["expected_sentiment"]).mean()
    chal_results.append({"Model": name, "Challenge_Accuracy": round(float(acc), 4)})

display(pd.DataFrame(chal_results))

# Explainability
engine = BoliSenseInferenceEngine()
test_input = "yo kaam ghna badiya konya"
explanation = engine.predict_and_explain(test_input)
print(f"\nInput: \"{test_input}\"")
print(f"Predicted: {explanation['prediction_label']} (Confidence: {explanation['confidence']:.1%})")
print(f"Net Logit Score: {explanation['net_logit']:.4f}")
print("\nTop Positive Drivers:")
for d in explanation["positive_contributors"][:3]:
    print(f"  + {d['feature']}: {d['contribution']:+.3f}")
print("Top Negative Drivers:")
for d in explanation["negative_contributors"][:3]:
    print(f"  - {d['feature']}: {d['contribution']:+.3f}")